In [ ]:
from upconversion_models import AndersonModel, AndersonModelUnfolded, cmap
from upconversion_models.jablonski_patch import (
    EmissionTransform,
    PulsedExcitation,
    Simulator,
    SteadyState,
)
from upconversion_models.transitions import PhononAssistedEnergyTransfer
from upconversion_models.utils import h, c, k
from poincare import solvers
from pint import get_application_registry
from scipy.optimize import curve_fit
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

u = get_application_registry()

Upconversion model with the 6th level unfolded into 6s (~540 nm) and 6h (~520 nm), extended with temperature dependent non-radiative relaxation and phonon-assisted energy transfer. Every temperature dependent rate is anchored at $T_0$ to Anderson's value, so the model reduces to Anderson's at $T_0$ and only departs from it when the temperature changes.

In [ ]:
M = AndersonModelUnfolded

sim = (
    Simulator(M())
    .with_solver(solvers.LSODA(rtol=1e-10, atol=1e-14))
    .with_transform(EmissionTransform(kind="emission"), append=True)
)

sim_and = (
    Simulator(AndersonModel())
    .with_solver(solvers.LSODA(rtol=1e-10, atol=1e-14))
    .with_transform(EmissionTransform(kind="emission"), append=True)
)

### Non-radiative relaxation

Each multiphonon relaxation $i \rightarrow j$ follows the energy-gap law, with $p = \Delta E_{ij} / \hbar\omega$ phonons and occupation $n(T) = 1/(e^{\hbar\omega/kT} - 1)$, scaled to Anderson's rate at $T_0$

$$
k_{i \rightarrow j}(T) = k_{i \rightarrow j}(T_0) \left[\frac{1 + n(T)}{1 + n(T_0)}\right]^p
$$

and comes with its upward partner fixed by detailed balance

$$
\frac{k_{j \rightarrow i}(T)}{k_{i \rightarrow j}(T)} = \frac{g_i}{g_j} \ e^{-\Delta E_{ij} / kT}
\quad \Rightarrow \quad
k_{j \rightarrow i}(T) = k_{i \rightarrow j}(T_0) \ \frac{g_i}{g_j} \left[\frac{n(T)}{1 + n(T_0)}\right]^p
$$

with $g = 2J + 1$. No new parameter is introduced.

### Phonon-assisted energy transfer

An energy transfer $\text{sensitizer}_{high} + \text{activator}_{low} \rightarrow \text{sensitizer}_{low} + \text{activator}_{high}$ is not resonant in general. The mismatch

$$
\Delta E = (E_{s,high} - E_{s,low}) - (E_{a,high} - E_{a,low})
$$

is compensated by emitting ($\Delta E > 0$) or absorbing ($\Delta E < 0$) $p$ phonons of energy $\varepsilon$ (Miyakawa–Dexter). Following Auzel (2003), large mismatches need multiphonon processes with the effective phonon energy, $\varepsilon = \hbar\omega$ and $p = |\Delta E| / \hbar\omega$, while mismatches smaller than $\hbar\omega$ are bridged by a single phonon of the mismatch energy, $\varepsilon = |\Delta E|$ and $p = 1$. With $n(T)$ the occupation of phonons of energy $\varepsilon$

$$
k(T) = k(T_0) \left[\frac{1 + n(T)}{1 + n(T_0)}\right]^p \quad (\Delta E > 0),
\qquad
k(T) = k(T_0) \left[r + (1 - r) \left(\frac{n(T)}{n(T_0)}\right)^p\right] \quad (\Delta E < 0)
$$

The resonant fraction $r$ would account for the Stark components and the inhomogeneous broadening that the level barycenters ignore. Here it is set to $r = 0$: the only endothermic transfers left after the next section are negligible.

### Yb$^{3+}$ crystal-field components

The $^2F_{5/2}$ level of Yb$^{3+}$ is split by the crystal field. In $\beta$-NaYF$_4$ its two lowest components $|0\rangle$ and $|1\rangle$ lie at 10 200 and 10 238 cm$^{-1}$ (Suyver 2006), so $\Delta_{01} = 38 \ cm^{-1}$. The transfer to the lowest component of Er$^{3+}$ $^4I_{11/2}$ is efficient from $|1\rangle$ (Suyver 2005), which Yu (2014) proposes as the origin of the increase of the emission between 10 K and 100 K. The two near-resonant Yb $\rightarrow$ Er steps, $^4I_{15/2} \rightarrow {}^4I_{11/2}$ (etu13) and $^4I_{11/2} \rightarrow {}^4F_{7/2}$ (etu37), are then weighted by the thermal population of $|1\rangle$

$$
p_1(T) = \frac{1}{1 + e^{\Delta_{01} / kT}},
\qquad
k(T) = k(T_0) \ \frac{\eta + (1 - \eta) \ p_1(T)}{\eta + (1 - \eta) \ p_1(T_0)}
$$

where $\eta$ is the relative efficiency of the transfer from $|0\rangle$. With $\eta = 0$ the upconversion collapses below ~30 K, as Suyver (2006) observed at 5 K.

In [ ]:
transfers = [
    name
    for name in dir(M)
    if isinstance(getattr(M, name, None), PhononAssistedEnergyTransfer)
]

mismatch = pd.DataFrame(
    {
        "mismatch [cm-1]": [
            (1 if getattr(M, t)._emits.default == 1 else -1)
            * getattr(M, t)._mismatch_wavenumber.default.to(1 / u.cm).magnitude
            for t in transfers
        ],
    },
    index=transfers,
)
hw = M.phonon_wavenumber.default.to(1 / u.cm).magnitude
mismatch["phonons p"] = [
    1 if getattr(M, t)._one_phonon.default else abs(dE) / hw
    for t, dE in mismatch["mismatch [cm-1]"].items()
]
mismatch["from Yb |1>"] = mismatch.index.isin(["etu13", "etu37"])
mismatch

### Anderson's compatibility

As in `paper.ipynb`, the transitions previously made by $n_6$ are now made by $n_s$ and $n_h$ combined

$$
k^X_6 = k^X_{6s} \ f_s(T_0) + k^X_{6h} \ (1 - f_s(T_0))
$$

so every radiative line of the unfolded model at $T_0$, with the lines leaving $6s$ and $6h$ recomposed into level $6$, must match Anderson's.

In [ ]:
LINES = [
    "rad", "rad81", "rad82", "rad83", "rad85", "rad61", "rad62", "rad63",
    "rad51", "rad52", "rad53", "rad31", "rad32", "rad2",
]


def recompose(df):
    # rad6sj + rad6hj -> rad6j
    out = df.copy()
    for j in (1, 2, 3):
        out[f"line_rad6{j}"] = out.pop(f"line_rad6s{j}") + out.pop(f"line_rad6h{j}")
    return out[[f"line_{line}" for line in LINES]]


fluxes = np.linspace(1, 100, 34) * u.W / u.cm**2

df_and = SteadyState().sweep(
    sim_and, variable=AndersonModel.energy_flux, values=fluxes
).to_dataframe()[[f"line_{line}" for line in LINES]]

df_unf = recompose(
    SteadyState()
    .sweep(sim.with_values({M.T: M.T0.default}), variable=M.energy_flux, values=fluxes)
    .to_dataframe()
)
df_unf.index = df_and.index

(df_unf / df_and - 1).abs().max().rename("max relative deviation at T0")

### Time resolved simulation against Anderson2013

In [ ]:
pulse = PulsedExcitation(
    pulse_width=10e-3 * u.s,
    pulse_height=2 * u.W / u.cm**2,
    pump="energy_flux",
)

save_at = np.linspace(0, 15e-3, 1000) * u.s
df = (
    pulse.solve(sim.with_values({M.T: M.T0.default}), save_at=save_at)
    .pint.dequantify()
    .to_dataframe()[["line_rad6s1", "line_rad6h1"]]
)

df = df.join(
    pulse.solve(sim_and, save_at=save_at)
    .pint.dequantify()
    .to_dataframe()[["line_rad61"]]
)

df.plot()

df["recomposition"] = df["line_rad6s1"] + df["line_rad6h1"]

plt.plot(df["recomposition"], lw=2, ls="--", color="gray", label="recomposition")
plt.legend()

### Yu's experimental results

Yu et al. (2014) measured $\beta$-NaYF$_4$:18%Yb,2%Er (Anderson's composition) between 10 K and 400 K, in vacuum, under 980 nm excitation of 2.9 W/mm$^2$. Below are the values for the bulk sample, read from their figures.

- Fig 4: emission intensity of 2H9/2 (blue), 4S3/2 (green) and 4F9/2 (red)
- Fig 5c: intensity ratio $R_{HS}$ of 2H11/2 to 4S3/2
- Fig 5d: intensity ratio $R_{r/g}$ of red to green
- Fig 7: rise and decay times after a 10 ns pulse at 980 nm

The red intensity of Fig 4c is not used: its red to green trend (decreasing ×4 between 200 K and 400 K) contradicts Fig 5d (increasing ×5) from the same paper.

In [ ]:
yu = pd.DataFrame(
    {
        "blue": [2.8e3, 2.5e3, 2.0e3, 6.7e3, 2.75e3, 1.7e3, 0.85e3, 0.35e3, 0.2e3, 0.1e3],
        "S": [2.9e4, 3.4e4, 3.65e4, 5.3e4, 4.25e4, 2.7e4, 2.4e4, 1.55e4, 1.1e4, 1.05e4],
        "R_HS": [0, 0, 0, 0, 0.02, 0.12, 0.23, 0.30, 0.58, 0.80],
        "R_rg": [0.21, 0.22, 0.22, 0.19, 0.20, 0.20, 0.31, 0.35, 0.68, 1.0],
    },
    index=pd.Index([10, 30, 50, 100, 150, 200, 250, 300, 350, 400], name="T"),
)

yu_lifetimes = pd.DataFrame(
    {
        "tauR_H9/2": [192, 198, 190, 180, 138, 112, 100, np.nan, np.nan, np.nan],
        "tauD_H9/2": [578, 582, 598, 578, 535, 510, 430, np.nan, np.nan, np.nan],
        "tauR_S3/2": [162, 190, 190, 162, 181, 139, 128, 88, 65, 45],
        "tauD_S3/2": [980, 940, 930, 895, 885, 795, 800, 720, 705, 700],
        "tauR_F9/2": [440, 478, 440, 397, 365, 402, 380, 395, 355, 352],
        "tauD_F9/2": [745, 850, 828, 805, 790, 650, 610, 445, 410, np.nan],
    },
    index=yu.index,
)  # µs, Fig 7 bulk
yu

### Calibration of the unfolded level with Yu's $R_{HS}$

Since $6s$ and $6h$ are thermalized, the intensity ratio follows Boltzmann

$$
R_{HS} = \frac{k^{rad}_{h} \ n_h}{k^{rad}_{s} \ n_s} = \frac{k^{rad}_{h}}{k^{rad}_{s}} \ \frac{g_h}{g_s} \ e^{-\Delta E / kT} = C \ e^{-\Delta E / kT}
$$

Fitting Yu's $R_{HS}$ gives $\Delta E$ (the energy of $6h$ above $6s$) and $k^{rad}_h / k^{rad}_s = C \ g_s / g_h$. Together with Anderson's constraint at $T_0$

$$
k^{rad}_6 = k^{rad}_{6s} \ f_s(T_0) + k^{rad}_{6h} \ (1 - f_s(T_0))
$$

both radiative rates are fixed.

In [ ]:
fit = yu[yu.index >= 200]
slope, intercept = np.polyfit(1 / fit.index, np.log(fit["R_HS"]), 1)

dE = (-slope * u.K * k / (h * c)).to(1 / u.cm)
C = np.exp(intercept)


def F(T):
    # n_h / n_s
    return 3 * np.exp(-1 * dE * h * c / (k * T))


fs0 = 1 / (1 + F(M.T0.default))
ratio = C / 3
ks = 1_510 / (fs0 + (1 - fs0) * ratio)
kh = ks * ratio

print(f"R_HS = {C:.2f} exp(-{-slope:.0f} K / T)  ->  dE = {dE:.0f~P}")
print(f"k_r6s = {float(ks):.0f} 1/s, k_r6h = {float(kh):.0f} 1/s")
print(
    "model defaults:",
    M.k_r6s.default,
    M.k_r6h.default,
    "Er6h - Er6s =",
    ((M.Er6h.energy - M.Er6s.energy) / (h * c)).to(1 / u.cm),
)

The phonon energy $\hbar\omega = 298 \ cm^{-1}$ is the dominant Raman mode of NaYF$_4$ (298, 370 and 418 cm$^{-1}$, Suyver et al. 2006). The same work obtains an effective 350 cm$^{-1}$ from the thermal shift and broadening of the Yb$^{3+}$ zero-phonon line; with 350 cm$^{-1}$ the lifetimes improve slightly, but the blue and green maxima are lower than Yu's. $\Delta_{01} = 38 \ cm^{-1}$ comes from the same work. The efficiency $\eta = 0.1$ of the transfer from Yb $|0\rangle$ is the only parameter calibrated against Yu's data (shapes of Fig 4a, 4b and 5d, and Fig 7).

### Temperature sweep against Yu2014

In [ ]:
Ts = yu.index.to_numpy() * u.K

ds = SteadyState().sweep(
    sim.with_values({M.energy_flux: 290 * u.W / u.cm**2}),
    variable=M.T,
    values=Ts,
)

df = ds.to_dataframe()
df.index = yu.index
df["blue"] = df["line_rad81"]
df["S"] = df["line_rad6s1"]
df["R_HS"] = df["line_rad6h1"] / df["line_rad6s1"]
df["R_rg"] = df["line_rad51"] / (df["line_rad6s1"] + df["line_rad6h1"])

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(9, 6), sharex=True)

for ax, col, color, label in [
    (axs[0, 0], "S", cmap["g"], "541 nm (relative to 300 K)"),
    (axs[0, 1], "blue", cmap["b"], "410 nm (relative to 300 K)"),
]:
    ax.plot(yu.index, yu[col] / yu.loc[300, col], "s", color=color, mfc="none", label="Yu")
    ax.plot(df.index, df[col] / df.loc[300, col], color=color, lw=2, label="simulated")
    ax.set_ylabel(label)
    ax.legend()

axs[1, 0].plot(yu.index, yu["R_HS"], "s", color="black", mfc="none", label="Yu")
axs[1, 0].plot(df.index, df["R_HS"], color="black", lw=2, label="simulated")
axs[1, 0].set_ylabel(r"$R_{HS}$")
axs[1, 0].legend()

axs[1, 1].plot(yu.index, yu["R_rg"] / yu.loc[300, "R_rg"], "s", color=cmap["r"], mfc="none", label="Yu")
axs[1, 1].plot(df.index, df["R_rg"] / df.loc[300, "R_rg"], color=cmap["r"], lw=2, label="simulated")
axs[1, 1].set_ylabel(r"$R_{r/g}$ (relative to 300 K)")
axs[1, 1].legend()

for ax in axs[1]:
    ax.set_xlabel("Temperature [K]")

plt.tight_layout()

Intensities and $R_{r/g}$ are compared relative to 300 K: the absolute values depend on the collection efficiency and spectral response of the setup. The absolute $R_{r/g}$ of the model is 2–4 times Yu's.

In [ ]:
cols = ["S", "blue", "R_rg"]
rel_model = df[cols] / df.loc[300, cols]
rel_yu = yu[cols] / yu.loc[300, cols]
np.log10(rel_model / rel_yu).pow(2).mean().pow(0.5).rename("rms log10 deviation")

### Lifetimes against Yu2014

After a 10 ns pulse at 980 nm, Yu fits each line with

$$
I(t) = I_1 \ e^{-t / \tau_D} - I_2 \ e^{-t / \tau_R}
$$

When rise and decay are close, this fit can not separate them. Instead, two quantities defined for any curve are compared: the time of the maximum, which for the function above is

$$
t_{max} = \frac{\tau_D \tau_R}{\tau_D - \tau_R} \ \ln \frac{\tau_D}{\tau_R}
$$

and the decay constant of the tail, $\tau_{tail} = \tau_D$. Yu does not report the pulse energy, so two fluences are simulated.

In [ ]:
def curve_metrics(t, y):
    # time of the maximum and exponential decay constant of the tail, in µs
    i = np.argmax(y)
    tail = (t > max(2 * t[i], t[i] + 50e-6)) & (y < 0.5 * y[i]) & (y > 1e-2 * y[i])
    slope = np.polyfit(t[tail], np.log(y[tail]), 1)[0]
    return t[i] * 1e6, -1e6 / slope


def t_max(tau_d, tau_r):
    return tau_d * tau_r / (tau_d - tau_r) * np.log(tau_d / tau_r)


transitions = {"H9/2": "line_rad81", "S3/2": "line_rad6s1", "F9/2": "line_rad51"}

yu_metrics = pd.DataFrame(index=yu.index)
for name in transitions:
    yu_metrics[f"tmax_{name}"] = t_max(yu_lifetimes[f"tauD_{name}"], yu_lifetimes[f"tauR_{name}"])
    yu_metrics[f"tail_{name}"] = yu_lifetimes[f"tauD_{name}"]

pulse_width = 10e-9 * u.s
save_at = np.linspace(0, 4e-3, 4001) * u.s

lifetimes = {}
for fluence in [10e-3, 30e-3] * u.J / u.cm**2:
    pulse = PulsedExcitation(
        pulse_width=pulse_width,
        pulse_height=fluence / pulse_width,
        pump="energy_flux",
    )
    rows = []
    for T in Ts:
        res = pulse.solve(sim.with_values({M.T: T}), save_at=save_at)
        res = res.pint.dequantify().to_dataframe()
        row = {}
        for name, line in transitions.items():
            tmax, tail = curve_metrics(res.index.to_numpy(), res[line].to_numpy())
            row[f"tmax_{name}"], row[f"tail_{name}"] = tmax, tail
        rows.append(row)
    lifetimes[f"{fluence.to('mJ/cm**2').magnitude:g} mJ/cm2"] = pd.DataFrame(rows, index=yu.index)

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(9, 9), sharex=True)

for i, (name, color) in enumerate([("H9/2", cmap["b"]), ("S3/2", cmap["g"]), ("F9/2", cmap["r"])]):
    for j, (metric, label) in enumerate([("tmax", r"$t_{max}$"), ("tail", r"$\tau_{tail}$")]):
        ax = axs[i, j]
        col = f"{metric}_{name}"
        ax.plot(yu.index, yu_metrics[col], "s", color=color, mfc="none", label="Yu")
        for (fluence, lt), ls in zip(lifetimes.items(), ["--", "-"]):
            ax.plot(lt.index, lt[col], color=color, ls=ls, lw=2, label=fluence)
        ax.set_ylabel(f"{label} {name} [µs]")
        ax.legend(fontsize=8)

for ax in axs[-1]:
    ax.set_xlabel("Temperature [K]")

plt.tight_layout()

In [ ]:
pd.DataFrame(
    {
        fluence: np.log10(lt / yu_metrics).pow(2).mean().pow(0.5)
        for fluence, lt in lifetimes.items()
    }
).rename_axis("rms log10 deviation")

### Remarks

- At $T_0$ every radiative line matches Anderson's within 0.25%.
- Every temperature dependence comes from the energy-gap law, detailed balance, phonon-assisted transfer and the thermal population of the Yb$^{3+}$ $|1\rangle$ component, with $\hbar\omega$ and $\Delta_{01}$ from Suyver (2006). Only $\eta$ is calibrated against Yu; the $6s$–$6h$ gap and $k^{rad}_{6s}, k^{rad}_{6h}$ follow from Yu's $R_{HS}$ and Anderson's constraint.
- Green (relative to 300 K): maximum of 3.4 at 100 K, as Yu's. The model is lower than Yu at 10 K (1.0 against 1.9) and at 400 K (0.35 against 0.68).
- Blue (relative to 300 K): ~5–6 between 10 K and 150 K, with a maximum of 6.0 at 50 K. Yu's bulk sample gives 6–8 with a single point of 19 at 100 K; the 25 nm and 45 nm samples show smooth maxima of ~7.
- Lifetimes at 30 mJ/cm$^2$: $t_{max}$ and $\tau_{tail}$ are within a factor ~1.3 (rms) for the green and the red and for the blue tail. The blue maximum arrives ~2 times later than Yu's at low temperature (695 against 317 µs at 10 K). The same freeze-out of $k_{nr8}$ that raises the blue intensity on cooling lengthens the 2H9/2 lifetime.
- The red tail at 300 K is longer than Yu's. In the model the red is limited by its feeding (6 → 9 → 5), while Yu's red tail matches the intrinsic 4F9/2 lifetime measured by Suyver (2006), 0.47 ms at 300 K.
- With $\hbar\omega = 350 \ cm^{-1}$ and $\eta = 0.3$ the lifetimes improve (blue maximum within a factor ~1.3), but the blue stays at ~4.5 and the green maximum at 2.7.
- With $\eta = 0$ no parameter is calibrated: the upconversion collapses below ~30 K, as Suyver (2006) observed at 5 K and unlike Yu's 10 K and 30 K points, which were measured under 290 W/cm$^2$ in vacuum and may be heated by the laser.
- Literature-backed changes that were tested and did not improve the agreement: the radiative/non-radiative split of 4F9/2 from Suyver (2006), the cross-relaxation $|^4S_{3/2}, ^4I_{15/2}\rangle \rightarrow |^4I_{9/2}, ^4I_{13/2}\rangle$ from Suyver (2006), a 2H9/2 → 4F3/2 gap of ~2000–2300 cm$^{-1}$ for $k_{nr8}$, and extending the Yb $|1\rangle$ weighting to the 6 → 9 and 5 → 8 transfers.
- The gap between $6s$ and $6h$ fitted from Yu's $R_{HS}$ (~520 cm$^{-1}$) is smaller than the spectroscopic one (650 ± 10 cm$^{-1}$, Suta 2025) and Tong's and Zhou's fits (~713 and ~752 cm$^{-1}$).